# conv-windowing-2d composite — cx14: build the 2D conv window view, then contract to a full Conv2d

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-windowing-2d`, `as-strided-windowing`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-windowing-2d"
DD_ATOM_IDS = ["conv-windowing-2d", "as-strided-windowing"]
DD_SUBTOPICS = ["CNN: 2-D conv windowing", "PyTorch: as_strided windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`conv-windowing-2d` is the 2-D specialization of `as-strided-windowing`. The generic atom says: 'to build a sliding-window view, the new axis's stride equals the source axis's stride'. Applied to the spatial `(H, W)` axes of an image tensor, this produces a `(B, IC, OH, OW, KH, KW)` tensor where each `(KH, KW)` patch is one kernel-sized window of the original image.

**The stride tuple.** For input strides `(s_b, s_ic, s_h, s_w)`:
```
x.as_strided(
    size=(B, IC, OH, OW, KH, KW),
    stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
)
```
The trailing pair `(s_h, s_w)` appears **twice** — once for the window-position axes (`OH, OW`) and once for the within-window axes (`KH, KW`). Same numerical stride, different roles.

**Contracting back to a full Conv2d.** Once you have the window view, the conv kernel `(OC, IC, KH, KW)` slots in as a tensor contraction:
```
einops.einsum(x_win, weight, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')
```
**Why two atoms.** The generic `as-strided-windowing` atom teaches the 'new-axis-stride-equals-source-stride' principle without committing to a 2-D shape. The CNN-specific atom is a direct application — but you should mentally derive it from the generic rule, not memorize the 6-tuple.

### Composite Exercise — build the 2D conv window view, then contract to a full Conv2d

**Atoms exercised together**: `conv-windowing-2d`, `as-strided-windowing`

Implement `cx14_conv2d_full(x, weight)` — the whitebox replacement for `F.conv2d(x, weight)` (stride=1, no padding).

- `x`: float tensor of shape `(B, IC, H, W)`.
- `weight`: float tensor of shape `(OC, IC, KH, KW)`.
- Return: tensor of shape `(B, OC, OH, OW)` where `OH = H - KH + 1`, `OW = W - KW + 1`.

1. **Apply the as_strided windowing rule** — read all four strides from `x.stride()`. The new axes' strides are the *source* spatial strides; the within-window axes' strides are the SAME source spatial strides.
2. **Build the 6-axis view** `(B, IC, OH, OW, KH, KW)` and confirm it shares storage with `x`.
3. **Einsum-contract** against the kernel.

The test cross-checks against `F.conv2d` and confirms the intermediate window view is a no-copy view.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx14_conv2d_full(x, weight):
    raise NotImplementedError

def cx14_window_view(x, KH, KW):
    """Return the (B, IC, OH, OW, KH, KW) strided view of x — pre-einsum."""
    raise NotImplementedError

def _test_cx14():
    from torch.nn import functional as F
    rng = t.Generator().manual_seed(14)

    # Case A: window-view storage sharing (no-copy property of as_strided).
    x = t.arange(1.0, 1 + 1*1*6*6).reshape(1, 1, 6, 6)
    win = cx14_window_view(x, KH=3, KW=3)
    assert tuple(win.shape) == (1, 1, 4, 4, 3, 3), f'window shape wrong: {tuple(win.shape)}'
    assert win.data_ptr() == x.data_ptr(), 'window view must share storage (no copy)'
    # Hand check: window (0,0) is x[0,0,:3,:3].
    assert t.allclose(win[0,0,0,0], x[0,0,:3,:3])
    # Window (2,1) is x[0,0,2:5,1:4].
    assert t.allclose(win[0,0,2,1], x[0,0,2:5,1:4])

    # Case B: full conv2d equivalence on multi-channel shapes.
    for B,IC,H,W,OC,KH,KW in [(2,3,14,16,4,5,3),(1,1,8,8,1,3,3),(3,2,12,18,6,3,5),(2,8,9,9,4,1,1)]:
        x2 = t.randn(B, IC, H, W, generator=rng)
        w2 = t.randn(OC, IC, KH, KW, generator=rng)
        yref = F.conv2d(x2, w2)
        yours = cx14_conv2d_full(x2, w2)
        assert tuple(yours.shape) == tuple(yref.shape)
        assert t.allclose(yours, yref, atol=1e-4), f'conv2d mismatch on {(B,IC,H,W,OC,KH,KW)}'

    # Case C: degenerate — KH==H, KW==W → single window per (b, ic).
    x3 = t.randn(2, 3, 5, 5, generator=rng)
    w3 = t.randn(4, 3, 5, 5, generator=rng)
    yours = cx14_conv2d_full(x3, w3)
    assert tuple(yours.shape) == (2, 4, 1, 1)
    assert t.allclose(yours, F.conv2d(x3, w3), atol=1e-4)
    _dd_passed.add('cx14')

_test_cx14()

<details><summary>Show solution — cx14</summary>

```python
def cx14_window_view(x, KH, KW):
    # Atom A (as-strided-windowing): new-axis strides = source spatial strides.
    B, IC, H, W = x.shape
    OH, OW = H - KH + 1, W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
    )

def cx14_conv2d_full(x, weight):
    # Atom B (conv-windowing-2d): wire window view into the conv einsum.
    OC, IC, KH, KW = weight.shape
    x_win = cx14_window_view(x, KH, KW)
    return einops.einsum(
        x_win, weight,
        'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow',
    )
```

The `(s_h, s_w, s_h, s_w)` block is the load-bearing piece. The first pair walks WINDOWS (one input row/column per window step). The second pair walks WITHIN a window (one input row/column per kernel cell). Same source strides, different semantic axes — that's the essence of the as-strided windowing trick.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["CNN: 2-D conv windowing", "PyTorch: as_strided windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()